## Type I - III diffusable nodes based on the Diego et al. 2018 using and Random matrix theory (RMT)

**Diffusion Rates:**

**Type I**   
DU, DV, DW = 1.0, 0.0, 10.0   (v immobile → this gives much higher values so lets do 1.0, 0, 1.0 ok so apparently we cant do that it has to be higher!!!!)  

**Type II**  
DU, DV, DW = 1.0, 1.0, 0.0    (w immobile)  

**Type III**  
DU, DV, DW = 0.0, 1.0, 1.0    (u immobile)  

Then for the LHS/robustness comparison, scan d from 0.1 to 10 and measure what fraction of our stable samples still gives Turing instability at each d value.

We should see Type I collapse to zero as d → 1, while Type II and III stay robustly non-zero...

In [36]:
import numpy as np
from numpy.linalg import eigvals
from scipy.optimize import fsolve
import matplotlib.pyplot as plt
from scipy.linalg import eig

In [37]:
# Adjacency Matrix

#       u  v  w
#     ┌─────────┐
#  u  │ 0  1  0 │  
#  v  │ 1  0  1 │
#  w  │ 1  1  1 │
#     └─────────┘

# The destabilizing module is the u-v mutual activation cycle: fuv * fvu > 0, this is the positive feedback loop that drives instability.

In [38]:
adjacency_matrix = np.array([
    [0, 1, 0],
    [1, 0, 1],
    [1, 1, 0],
])

# the diagonal of the adjacency should always be 0
# the w self-loop [2,2] is already handled by J = G - I giving diagonal = -1.
# self-decay handled by J = G - I
# adjacency_matrix[i,j] = 1 means species j affects species i

### Type I

_v is immobile, w and u > 1_

In [51]:
# apply sign constraints for specific topology and type (activating (+) or inhibiting (-))
# for type I, the u-v cycle is destabilising and the v-w cycle is stabilising, so we have:

def sign_constraints(J):
    J[0, 1] =  abs(J[0, 1]) # v activates u, u-v destabilising
    J[1, 0] =  abs(J[1, 0]) # u activates v
    J[1, 2] =  abs(J[1, 2]) # w activates v
    J[2, 1] = -abs(J[2, 1]) # v inhibits w, v-w cycle must be stabilising → edges opposite sign
    # J[2, 0] unconstrained

    return J

# template:
# J[i, j] =  abs(J[i, j])   # j activates i
# J[i, j] = -abs(J[i, j])   # j inhibits i
# leave unconstrained edges untouched

# so we don't do: J = G - np.eye(3), because it has no sign constraints (want to enforce the sign constraints for the specific topology)

In [52]:
def generate_jacobian_type1(sigma):
    
    # random matrix
    G = np.random.normal(0, sigma, (3, 3))
    np.fill_diagonal(G, 0)
    
    # J = G - I  (rmt convention, may 1972), diagonal becomes -1, self-decay handled by J = G - I, off-diagonal from N(0, sigma)
    J = G - np.eye(3)
    
    # apply sparsity mask after sampling from adjacency matrix, but only to off-diagonal elements, the diagonal is already -1 from J = G - I
    for i in range(3): 
        for j in range(3):
            if i != j and adjacency_matrix[i, j] == 0:
                J[i, j] = 0

    J = sign_constraints(J)

    return J

In [53]:
def is_stable(J):
    return np.all(np.real(eigvals(J)) < 0)

In [54]:
# check turing instabilitiy: does diffusion destabilise a mode that was stable without diffusion?

# two ways to check turing stability:
# 1. shaberi et al (2025)
#    - compute all eigenvalues and check if real part < 0
#    - detects any instability, including oscillatory ones (complex eigenvalues)
#    - detects Turing I only (defined wavelength, restabilises at large k)
# 2. diego et al (2018) 
#    - characteristic polynomial and Routh-Hurwitz criteria
#    - detects only stationary instabilities, not oscillary ones
#    - condition: a3 < 0 AND a1 > 0 AND a2 > 0 for some k > 0.

def is_turing_shaberi(J, DU, DV, DW):
    D = np.diag([DU, DV, DW])
    any_unstable = False
    for k in np.linspace(0.1, 10, 100):                        # changed np.arange(0.1, 10, 100) to np.linspace(0.1, 10, 100) bc long run time
        M = J - k**2 * D
        if np.max(np.real(eigvals(M))) > 0:
            any_unstable = True
            break
    
    if not any_unstable:
        return False
    
    M_large = J - 10**2 * D
    return np.all(np.real(eigvals(M_large)) < 0)

def is_turing_diego(J, DU, DV, DW):
    D = np.diag([DU, DV, DW])
    for k in np.logspace(-1, 2, 100):                          # changed (-1, 2, 500) to (-1, 2, 100) bc long run time
        M  = J - k**2 * D
        a1 = -np.trace(M)
        a2 = (M[0,0]*M[1,1] - M[0,1]*M[1,0] +
              M[0,0]*M[2,2] - M[0,2]*M[2,0] +
              M[1,1]*M[2,2] - M[1,2]*M[2,1])
        a3 = -np.linalg.det(M)
        if a3 < 0 and a1 > 0 and a2 > 0:
            return True
    return False

In [55]:
# type I specifications

n_samples = 100_000
#sigma = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.0]
sigma = [0.6, 0.7]
DU, DV, DW = 1.0, 0.0, 10.0

In [56]:
results_type1 = []

for sig in sigma:
    np.random.seed(42)
    stable = 0
    turing_diego = 0
    turing_shaberi = 0

    for _ in range(n_samples):
        J = generate_jacobian_type1(sig)
        if is_stable(J):
            stable += 1
            if is_turing_diego(J, DU, DV, DW):
                turing_diego += 1
            if is_turing_shaberi(J, DU, DV, DW):
                turing_shaberi += 1

    rob_diego   = 100 * turing_diego   / stable if stable > 0 else 0.0
    rob_shaberi = 100 * turing_shaberi / stable if stable > 0 else 0.0

    results_type1.append({
        "sigma":        sig,
        "stable":       stable,
        "diego":        turing_diego,
        "shaberi":      turing_shaberi,
        "rob_diego":    rob_diego,
        "rob_shaberi":  rob_shaberi,
    })

print(f"{'Sigma':<8} {'Tested':>8} {'Stable':>8} {'Diego_Tu':>10} {'Shaberi_Tu':>12} {'Diego_Ro':>11} {'Shaberi_Ro':>11}")
print("-" * 80)

for r in results_type1:
    print(f"{r['sigma']:<6.1f} {n_samples:>10,} {r['stable']:>7} {r['diego']:>8} {r['shaberi']:>12} "
          f"{r['rob_diego']:>14.7f}% {r['rob_shaberi']:>14.7f}%")

Sigma      Tested   Stable   Diego_Tu   Shaberi_Tu    Diego_Ro  Shaberi_Ro
--------------------------------------------------------------------------------
0.6       100,000   97740      104          101      0.1064047%      0.1033354%
0.7       100,000   94897      320          315      0.3372077%      0.3319388%


### Type II

_w is immobile, u and v > 1_

for type II the destabilizing module switches from the u-v to the v-w pair!!

In [ ]:
# apply sign constraints for specific topology and type (activating (+) or inhibiting (-))
# for type I, the u-v cycle is destabilising and the v-w cycle is stabilising, so we have:

def sign_constraints(J):
    J[1, 2] =  abs(J[1, 2])   # w activates v
    J[2, 1] =  abs(J[2, 1])   # v activates w  → v-w cycle positive = destabilising
    J[0, 1] =  abs(J[0, 1])   # v activates u
    J[1, 0] = -abs(J[1, 0])   # u inhibits v   → u-v cycle negative = stabilising
    return J

In [ ]:
def generate_jacobian_type2(sigma):
    
    # random matrix
    G = np.random.normal(0, sigma, (3, 3))
    np.fill_diagonal(G, 0)
    
    # J = G - I  (rmt convention, may 1972), diagonal becomes -1, self-decay handled by J = G - I, off-diagonal from N(0, sigma)
    J = G - np.eye(3)
    
    # apply sparsity mask after sampling from adjacency matrix, but only to off-diagonal elements, the diagonal is already -1 from J = G - I
    for i in range(3): 
        for j in range(3):
            if i != j and adjacency_matrix[i, j] == 0:
                J[i, j] = 0

    J = sign_constraints(J)

    return J

In [ ]:
def is_stable(J):
    return np.all(np.real(eigvals(J)) < 0)

In [ ]:
# check turing instabilitiy: does diffusion destabilise a mode that was stable without diffusion?

# two ways to check turing stability:
# 1. shaberi et al (2025)
#    - compute all eigenvalues and check if real part < 0
#    - detects any instability, including oscillatory ones (complex eigenvalues)
#    - detects Turing I only (defined wavelength, restabilises at large k)
# 2. diego et al (2018) 
#    - characteristic polynomial and Routh-Hurwitz criteria
#    - detects only stationary instabilities, not oscillary ones
#    - condition: a3 < 0 AND a1 > 0 AND a2 > 0 for some k > 0.

def is_turing_shaberi(J, DU, DV, DW):
    D = np.diag([DU, DV, DW])
    any_unstable = False
    for k in np.arange(0.01, 10, 0.01):
        M = J - k**2 * D
        if np.max(np.real(eigvals(M))) > 0:
            any_unstable = True
            break
    
    if not any_unstable:
        return False
    
    M_large = J - 10**2 * D
    return np.all(np.real(eigvals(M_large)) < 0)

def is_turing_diego(J, DU, DV, DW):
    D = np.diag([DU, DV, DW])
    for k in np.logspace(-1, 2, 500):
        M  = J - k**2 * D
        a1 = -np.trace(M)
        a2 = (M[0,0]*M[1,1] - M[0,1]*M[1,0] +
              M[0,0]*M[2,2] - M[0,2]*M[2,0] +
              M[1,1]*M[2,2] - M[1,2]*M[2,1])
        a3 = -np.linalg.det(M)
        if a3 < 0 and a1 > 0 and a2 > 0:
            return True
    return False

In [ ]:
# type II specifications

n_samples = 100000
#sigma = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.0]
sigma = [0.6, 0.7]
DU, DV, DW = 1.0, 1.0, 0.0

In [ ]:
results_type2 = []

for sig in sigma:
    np.random.seed(42)
    stable = 0
    turing_diego = 0
    turing_shaberi = 0

    for _ in range(n_samples):
        J = generate_jacobian_type2(sig)
        if is_stable(J):
            stable += 1
            if is_turing_diego(J, DU, DV, DW):
                turing_diego += 1
            if is_turing_shaberi(J, DU, DV, DW):
                turing_shaberi += 1

    rob_diego   = 100 * turing_diego   / stable if stable > 0 else 0.0
    rob_shaberi = 100 * turing_shaberi / stable if stable > 0 else 0.0

    results_type2.append({
        "sigma":        sig,
        "stable":       stable,
        "diego":        turing_diego,
        "shaberi":      turing_shaberi,
        "rob_diego":    rob_diego,
        "rob_shaberi":  rob_shaberi,
    })

print(f"{'sigma':<8} {'stable':>8} {'diego_n':>10} {'shaberi_n':>12} {'diego_%':>10} {'shaberi_%':>12}")
print("-" * 65)

for r in results_type2:
    print(f"{r['sigma']:<8.1f} {r['stable']:>8} {r['diego']:>10} {r['shaberi']:>12} "
          f"{r['rob_diego']:>9.3f}% {r['rob_shaberi']:>11.3f}%")

### Type III

_u is immobile, w and v are mobile, w > 0 !!_